# 1. Instalasi Library

In [1]:
import os
import re
import glob
import time 
import shutil
# import warnings
from collections import Counter

import numpy as np
import pandas as pd
import scipy.stats as sp
import matplotlib.pyplot as plt
from IPython.display import display

# warnings.filterwarnings("ignore")
# pd.set_option("display.max_column", 40)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 220)
plt.rcParams["figure.dpi"] = 110

SEED = 42
np.random.seed(SEED)
FOLDER_HASIl = "Hasil Perbandingan"
os.makedirs("FOLDER_HASIL", exist_ok=True)

# 2. Konfigurasi

In [2]:
DATASET = "/kaggle/input/datasets/eaganrainmahasin/dataset-inverted-csv"
KOLOM_ID = ["docID"]
KOLOM_JUDUL = ["title"]
KOLOM_TEKS = ["article_text"]
KOLOM_TOKEN = ["tokens_stemmed"]
MAKS_DOKUMEN = None

K1_BAWAAN = 1.2
B_BAWAAN = 0.75
TOP_K = 10

K_PERCOBAAN = [0.5, 1.2, 2.0, 3.0]
B_PERCOBAAN = [0.0, 0.25, 0.5, 0.75, 1.0]

DAFTAR_QUERY = [
    {"teks": "pendaftaran petugas KPPS Pemilu 2024", "relevan": ["26", "64", "480", "190", "41", "315", "793", "278", "467", "683", "662", "2", "10", "348", "728"]},
    {"teks": "gaji petugas KPPS", "relevan": ["64", "41", "480", "26"]},
    {"teks": "Prabowo naik helikopter TNI ke Marapi", "relevan": ["27", "8"]},
    {"teks": "partai rekomendasi Khofifah cagub Jatim", "relevan": ["55", "78", "221", "846", "30", "12", "60", "874", "863", "335", "871", "870", "147"]},
    {"teks": "laporan dana kampanye parpol", "relevan": ["67", "54", "33"]},
    {"teks": "pengamanan Natal dan Tahun Baru", "relevan": ["3", "489", "437", "474"]},
    {"teks": "panelis dan moderator debat capres", "relevan": ["113", "573", "101", "181", "15", "95", "169", "174", "454", "714", "611"]},
    {"teks": "Gibran kunjungi pondok pesantren", "relevan": ["912", "117", "925", "14", "856", "881", "872", "257"]},
    {"teks": "politik dinasti Ade Armando Jogja", "relevan": ["471", "482", "499", "535", "582", "684", "741", "786", "800", "802", "804", "812", "816", "852", "866", "869", "883", "890", "891", "913", "917", "926", "933", "934", "935", "945", "946"]},
    {"teks": "cek NIK terdaftar anggota parpol", "relevan": ["18"]},
    {"teks": "regulasi kelautan nelayan Ganjar", "relevan": ["29"]},
    {"teks": "daerah rawan pemilu pengamanan polisi", "relevan": ["19"]},
    {"teks": "gugatan pencalonan Gibran", "relevan": ["17"]},
    {"teks": "pemilih muda milenial gen z", "relevan": ["632", "483", "999", "756", "153", "85", "20", "610"]},
]

# 3. Memuat Korpus

In [3]:
#mencari file csv
file_csv = glob.glob('/kaggle/input/**/*.csv', recursive=True)

if file_csv:
    print("file CSV ditemukan:")
    for file in file_csv:
        print(file)
else:
    raise FileNotFoundError("Tidak ada file CSV yang ditemukan!")

path_csv = file_csv[0]

#membaca csv
df = None
for enc in ["utf-8", "utf-8-sig", "latin-1"]:
    try:
        df = pd.read_csv(path_csv, encoding=enc, on_bad_lines="skip")
        break
    except UnicodeDecodeError:
        continue

if df.shape[1]==1:
    df = pd.read_csv(path_csv, encoding=enc, sep=None, engine="python", on_bad_lines="skip")

print("\n", "+="*10, "INFORMASI SKORPUS", "+="*10)
print("Jumlah baris:", df.shape[0])
print("Jumlah kolom:", df.shape[1])
print("Daftar kolom:", df.columns)

display(df.head())

file CSV ditemukan:
/kaggle/input/datasets/eaganrainmahasin/dataset-inverted-csv/dataset_inverted.csv

 +=+=+=+=+=+=+=+=+=+= INFORMASI SKORPUS +=+=+=+=+=+=+=+=+=+=
Jumlah baris: 1000
Jumlah kolom: 4
Daftar kolom: Index(['docID', 'title', 'article_text', 'tokens_stemmed'], dtype='object')


,docID,title,article_text,tokens_stemmed
0,0,Hasto Ingin Debat Pilpres Pertarungan Gagasan dan Tak Ada Rekayasa,Sekjen PDIP sekaligus Sekretaris Tim Pemenang Nasional (TPN) Ganja...,"['sekjen', 'pdip', 'sekaligus', 'sekretaris', 'tim', 'menang', 'na..."
1,1,"Terima Kunjungan Gibran di Ponpes Al-Tsaqafah, Said Aqil Tegaskan ...",Mantan Ketua Umum Pengurus Besar Nahdlatul Ulama (PBNU) Said Aqil ...,"['mantan', 'ketua', 'umum', 'urus', 'besar', 'nahdlatul', 'ulama',..."
2,2,Contoh Format dan Isi Surat Pernyataan Anggota KPPS Pemilu 2024,Surat pernyataan KPPS Pemilu 2024 merupakan bagian dari berkas ata...,"['surat', 'nyata', 'kpps', 'milu', 'rupa', 'bagi', 'berkas', 'doku..."
3,3,9 Ribu Personel Gabungan Dikerahkan Amankan Natal-Tahun Baru di Sumut,Sekitar 9.000 personel dikerahkan untuk mengamankan perayaan Natal...,"['personel', 'kerah', 'aman', 'raya', 'natal', 'tahun', 'baru', 's..."
4,4,Amarah Ukraina Usai Rusia Bakal Gelar Pilpres di Wilayah Pendudukan,"Ukraina marah usai Rusia menjadikan empat wilayahnya, yang kini di...","['ukraina', 'marah', 'usai', 'rusia', 'jadi', 'empat', 'wilayah', ..."
